# **Actividad 1. Exploración de un proxy climático y detección de cambios abruptos**

# Loading and Visualization

In this section, we illustrate how to load and visualize the pseudoPAGES2k dataset with `cfr`.

Required data to complete this tutorial:

- pseudoPAGES2k: [ppwn_SNRinf_rta.nc](https://github.com/fzhu2e/paper-pseudoPAGES2k/raw/main/data/ppwn_SNRinf_rta.nc)

In [ ]:
%load_ext autoreload
%autoreload 2

%pip install cartopy
%pip install cfr
%pip install requests
%pip install nbformat>=4.2.0

import cfr
print(cfr.__version__)
import xarray as xr

## Load the pseudoPAGES2k dataset with `xarray`

By default, we may load a netCDF file with `xarray` to have a check of the data structure:

In [ ]:
ds = xr.open_dataset('./data/ppwn_SNRinf_rta.nc', use_cftime=True)
ds

In [ ]:
ds['Arc_076']

We see that this netCDF file has multiple data variables named after **proxy IDs**.
Each variable comes with the below fundamental attributes:

- lat: the latitude of the site
- lon: the longitude of the site
- ptype: the proxy type
- dt: step of the time axis, i.e., temporal resolution
- time_name: name of the time axis
- time_unit: unit of the time axis
- value_name: unit of the value axis
- value_unit: unit of the value axis

With this certain format, the netCDF file is `cfr` ready.

## Load the pseudoPAEGS2k dataset with `cfr`

The `cfr.ProxyDatabase` class comes with a `.load_nc()` method that can help us load a proxy database from a netCDF file following the certain format shown above:

In [ ]:
# load the pseudoPAGES2k database from a netCDF file

# load from a local copy
# pdb = cfr.ProxyDatabase().load_nc('./data/ppwn_SNRinf_rta.nc')

# load from the cloud
pdb = cfr.ProxyDatabase().fetch('pseudoPAGES2k/ppwn_SNRinf_rta')

## Visualize and check the pseudoPAGES2k dataset by using **proxy IDs** on an interactive map

One may ask "How do I know the proxy IDs?"

Once the netCDF file is loaded as a `cfr.ProxyDatabase`, we can easily visualize the dataset with the `.plot().

A `cfr.ProxyDatabase` object also comes with a `.plotly()` method that can help us check **proxy IDs** on an interactive map.
It will display an interactive map, and by hovering the mouse over each site marker, one may check the metadata of a specific site, including:

- pid (proxy ID)
- ptype
- lat
- lon

In [ ]:
pdb.plotly()

We may also plot map along with the count of the records:

In [ ]:
fig, ax = pdb.plot(plot_count=True)

Since the dataset starts from 850 AD, we may adjust the x-axis utilizing the `matplotlib` methods:

In [ ]:
fig, ax = pdb.plot(plot_count=True)
ax['count'].set_xlim(800, 2000)

## Access and visualize a specific record

A specific record can be accessed by its **proxy ID**:

In [ ]:
pobj = pdb['Arc_076']
pobj

This returned object is defined by `cfr.ProxyRecord`, which comes with several attributes such as:
- time: the time axis
- value: the value axis
- lat: the latitude of the site
- lon: the longitude of the site
- (... other metadata)

For instance, to access the record series:

In [ ]:
print('time axis:', pobj.time)
print('value axis:', pobj.value)

Now that we have the `cfr.ProxyRecord` object, we can easily visualize the record utilizing the `.plot()` method:

In [ ]:
fig, ax = pobj.plot()

We may **slice the record** to zoom in and out.
For instance, let us check the instrumental period:

In [ ]:
fig, ax = pobj.slice([1500, 1700]).plot()

We may also slice with a shortcut using strings of years:

In [ ]:
fig, ax = pobj['1850':'2000'].plot()

This shortcut also supports time step.
For instance, to display the data points every 10 years between 1850 AD and 2000 AD:

In [ ]:
fig, ax = pobj['1850':'2000':'5'].plot(marker='o')

# **Actividad 2. Del proxy al clima: calibración proxy–clima mediante un PSM**

# Proxy System Model (PSM) by Univariate linear regression

It is a mathematical or physical model used to understand how environmental changes are recorded by a geological or biological archive.

In this section, we introduce the default PSM in `cfr` based on univariate linear regression.
It can be applied to any proxy type that is believed to have a univariate linear relationship with a certain climate variable.
It also supports a seasonality searching procedure to help determine the seasonality of a specific site.

For instance, this PSM can be applied to the `tree.MXD` records, which we believe have high linear correlation with the local temperature condition over a growing season.

In [ ]:
pobj = pdb.records['Arc_076']
fig, ax = pobj.plot()

### Model

Its job is to mathematically simulate how one single environmental variable (usually temperature or precipitation) translates into a physical proxy measurement (like tree-ring width or coral $\delta^{18}\text{O}$).

The Mathematical FrameworkIn a standard univariate linear PSM, the proxy value is modeled as a straight-line function of the climate variable, plus random environmental/measurement noise:
$$P_{t} = \beta_{0} + \beta_{1} \cdot C_{t} + \epsilon$$

where,
- $P_{t}$: The predicted Proxy value at time
- $C_{t}$: The input Climate variable (e.g., local surface temperature).
- $\beta _{0}$: The intercept (the base state of the proxy).
- $\beta _{1}$: The slope (the sensitivity of the proxy to the climate change).
- $\epsilon$: The residual error (noise), representing non-climatic influences


##### Load climate-model fields

Load model temperature and precipitation fields from the iCESM past1000 historical simulation.

The iCESM (Isotope-enabled Community Earth System Model) simulations for the past1000 (Last Millennium, 850–1850 CE) and historical (1850–2005 CE) periods are key climate datasets. They trace how water isotopes (δ¹⁸O and δD) move through the global atmosphere, land, ocean, and sea ice. This helps scientists calibrate climate models directly against physical paleoclimate proxies like ice cores, speleothems (cave formations), and tree rings.

In [ ]:
model_tas = cfr.ClimateField().fetch('iCESM_past1000historical/tas')
model_pr = cfr.ClimateField().fetch('iCESM_past1000historical/pr')

Load model temperature and precipitation fields from the iCESM past1000 historical simulation.

In [ ]:
model_tas.da

In [ ]:
import numpy as np

print(np.median(np.diff(model_tas.da.lat)))
print(np.median(np.diff(model_tas.da.lon)))

### Instrumental observations

CRUTSv4.07 (Climatic Research Unit Time-Series, version 4.07) is a high-resolution gridded climate dataset that tracks global month-by-month climate variations from January 1901 to December 2022 https://crudata.uea.ac.uk/cru/data/hrg/cru_ts_4.07/

Key SpecificationsSpatial Resolution: 
- 0.5° latitude by 0.5° longitude grid, covering all global land domains except Antarctica.
- Temporal Coverage: Monthly historical data covering 122 years (1901–2022).
- Methodology: It utilizes an Angular-Distance Weighting (ADW) interpolation method to turn raw weather station observations into a complete, seamless global grid.

In [ ]:
obs_tas = cfr.ClimateField().fetch('CRUTSv4.07/tas', vn='tmp')
obs_pr = cfr.ClimateField().fetch('CRUTSv4.07/pr', vn='pre')

#### Standardize observational variable names

Rename the downloaded observational variables to `tas` for temperature and `pr` for precipitation.

In [ ]:
import numpy as np

obs_tas = obs_tas.rename('tas')
obs_pr = obs_pr.rename('pr')

Visualize the CRUTSv4.07 temperature field using contour levels from −40 °C to 40 °C.

In [ ]:
fig, ax = obs_tas.plot(levels=np.linspace(-40, 40, 20), cmap='RdBu_r')

### Inspect temperature coordinates

Display the original coordinates of the observational temperature field.

In [ ]:
obs_tas_new = obs_tas.wrap_lon()

Verify the coordinates after adjusting the longitude convention.

In [ ]:
obs_tas_new.da.coords

In [ ]:
fig, ax = obs_tas_new.plot(levels=np.linspace(-40, 40, 20), cmap='RdBu_r')

### Inspect precipitation coordinates

Display the original coordinates of the observational precipitation field.

In [ ]:
fig, ax = obs_pr.plot(levels=np.linspace(-40, 40, 20), cmap='Blues_r')

#### Wrap precipitation longitudes

Convert the precipitation field longitudes to the same convention as the temperature field.

In [ ]:
obs_pr_new = obs_pr.wrap_lon()

Verify the precipitation coordinates after longitude adjustment.

In [ ]:
obs_pr_new.da.coords

## Get climate data for a specific `ProxyRecord`

In [ ]:
%%time

pobj.del_clim()
pobj.get_clim(model_tas, tag='model')
pobj.get_clim(model_pr, tag='model')
pobj.get_clim(obs_tas_new, tag='obs')
pobj.get_clim(obs_pr_new, tag='obs')

In [ ]:
pobj.clim['obs.tas'].da

## Create a PSM object

#### Calibrate the linear PSM

Test the candidate seasonal windows and calibrate the univariate linear proxy system model.

In [ ]:
lr_mdl = cfr.psm.Linear(pobj)

In [ ]:
%%time
sn_list = [
    [1,2,3,4,5,6,7,8,9,10,11,12],
    [6,7,8],
    [3,4,5,6,7,8],
    [6,7,8,9,10,11],
    [-12,1,2],
    [-9,-10,-11,-12,1,2],
    [-12,1,2,3,4,5]
]
lr_mdl.calibrate(season_list=sn_list)

#### Review calibration details

Inspect the fitted model parameters and diagnostics produced during calibration.

In [ ]:
lr_mdl.calib_details

#### Generate the modeled proxy

Use the calibrated PSM to forward-model the proxy response from the climate data.

In [ ]:
%%time
pp = lr_mdl.forward()

#### Plot the modeled proxy series

Visualize the proxy series generated by the calibrated linear PSM.

In [ ]:
fig, ax = pp.plot()

# **Actividad 3. Del conjunto de proxies a la reconstrucción climática espacial**

# Reconstructing the tropical Pacific SST with PAGES2k and GraphEM

**Expected time to run through: ~0.5 hrs**

In this section, we illustrate the basic workflow of the Graphical Expectation-Maximization algorithm (GraphEM, [Guillot et al., 2015](https://doi.org/10.1214/14-AOAS794)) with `cfr`, conducting a reconstruction experiment with the PAGES2k dataset.
Due to the intensive computational requirement of the GraphEM method, our goal is to reconstruct the air surface temperature field only over the tropical Pacific region using coral records.

Note: `pip install "cfr[graphem]"` to enable the GraphEM method if not done before.

In [ ]:
# create a reconstruction job object using the `cfr.ReconJob` class
job = cfr.ReconJob()

# load the pseudoPAGES2k database from a netCDF file

# load from a local copy
# job.proxydb = cfr.ProxyDatabase().load_nc('./data/ppwn_SNRinf_rta.nc')

# load from the cloud
job.load_proxydb('PAGES2kv2')

# filter the database
job.filter_proxydb(by='ptype', keys='coral')

# plot to have a check of the database
fig, ax = job.proxydb.plot(plot_count=True)
ax['count'].set_xlim(800, 2000)

### Annualize each proxy record

In [ ]:
job.annualize_proxydb(months=[12, 1, 2], verbose=True)

In [ ]:
job.load_clim(
    tag='obs',
    path_dict={
        'tas': 'gistemp1200_GHCNv4_ERSSTv5', # load from the cloud
    },
    rename_dict={'tas': 'tempanomaly'},
    anom_period=(1951, 1980),
    load=True,  # load the data into memeory to accelerate the later access; requires large memeory
    verbose=True,
)

# Reconstructing the tropical Pacific SST with PAGES2k and GraphEM

**Expected time to run through: ~0.5 hrs**

In this section, we illustrate the basic workflow of the Graphical Expectation-Maximization algorithm (GraphEM, [Guillot et al., 2015](https://doi.org/10.1214/14-AOAS794)) with `cfr`, conducting a reconstruction experiment with the PAGES2k dataset.
Due to the intensive computational requirement of the GraphEM method, our goal is to reconstruct the air surface temperature field only over the tropical Pacific region using coral records.

Note: `pip install "cfr[graphem]"` to enable the GraphEM method if not done before.

In [ ]:
#python -m pip install --upgrade pip setuptools wheel
#python -m pip install "cfr[graphem]"

In [ ]:
#%reload_ext autoreload
#%autoreload 2

import cfr
import numpy as np

print(cfr.__version__)

In [ ]:
%load_ext autoreload
%autoreload 2
%pip install "cfr[graphem]"

import cfr
print(cfr.__version__)
import numpy as np

## GraphEM steps
### Create a reconstruction job object `cfr.ReconJob` and load the pseudoPAGES2k database
      
A `cfr.ReconJob` object takes care of the workflow of a reconstruction task.
It provides a series of attributes and methods to help the users go through each step of the reconstruction task, such as loading the proxy database, loading the model prior, calibrating and running the proxy system models, performing the data assimilation solver, etc.

In [ ]:
# create a reconstruction job object using the `cfr.ReconJob` class
job = cfr.ReconJob()

# load the pseudoPAGES2k database from a netCDF file

# load from a local copy
# job.proxydb = cfr.ProxyDatabase().load_nc('./data/ppwn_SNRinf_rta.nc')

# load from the cloud
job.load_proxydb('PAGES2kv2')

# filter the database
job.filter_proxydb(by='ptype', keys='coral')

# plot to have a check of the database
fig, ax = job.proxydb.plot(plot_count=True)
ax['count'].set_xlim(800, 2000)

### Annualize each proxy record

In [ ]:
job.annualize_proxydb(months=[12, 1, 2], verbose=True)

### Load the instrumental observations

As a perfect model prior pseudoproxy experiment (PPE), we use the iCESM simulated fields as instrumental observations.

In [ ]:
job.load_clim(
    tag='obs',
    path_dict={
        'tas': 'gistemp1200_GHCNv4_ERSSTv5', # load from the cloud
    },
    rename_dict={'tas': 'tempanomaly'},
    anom_period=(1951, 1980),
    load=True,  # load the data into memeory to accelerate the later access; requires large memeory
    verbose=True,
)

### Annualize the observation fields

This step will determine the temporal resolution of the reconstructed fields.

In [ ]:
job.annualize_clim(tag='obs', verbose=True, months=[12, 1, 2])

### Regrid the observation fields

This step will determine the spatial resolution of the reconstructed fields.

In [ ]:
job.regrid_clim(tag='obs', nlat=42, nlon=63, verbose=True)

### Crop the observations fields to make the problem size smaller

In [ ]:
job.crop_clim(tag='obs', lat_min=-20, lat_max=20, lon_min=150, lon_max=260, verbose=True)

In [ ]:
# check the cropped domain
fig, ax = job.obs['tas'][-1].plot()

### (Optional) Save the job object for later reload

Save the job object before running the DA procedure for a quick reload next time if needed.

In [ ]:
job.save('./cases/graphem-real-pages2k', verbose=True)

Now let's reload the job object from the saved directory.

In [ ]:
job = cfr.ReconJob()
job.load('./cases/graphem-real-pages2k/', verbose=True)

### Prepare the GraphEM solver

In [ ]:
job.prep_graphem(
    recon_period=(1871, 2000),  # period to reconstruct
    calib_period=(1901, 2000),  # period for calibration
    uniform_pdb=True,           # filter the proxydb to be more uniform
    verbose=True,
)

### Run the GraphEM solver

We will take the Empirical Graphs (graphical lasso, `glasso`) approach.

In [ ]:
job.run_graphem(
    save_dirpath='./recons/graphem-real-pages2k',
    graph_method='hybrid',
    cutoff_radius=5000,
    sp_FF=2, sp_FP=2,
    verbose=True,
)

## Validation steps

### Create the reconstruction result object `cfr.ReconRes`.

A `cfr.ReconRes` object takes care of the workflow of postprocessing and analyzing the reconstruction results.
It provides handy methods to help the users load, validate, and visualize the reconstruction results.

In [ ]:
res = cfr.ReconRes('./recons/graphem-real-pages2k', verbose=True)

### Load the reconstructed variables

Here we validate the `tas` field and the NINO3.4 index as an example.

In [ ]:
res.load(['tas', 'nino3.4'], verbose=True)

### Validate the reconstructed NINO3.4

We calculate the annualized NINO3.4 from Bunge & Clarke (2009) as a reference for validation.

In [ ]:
bc09 = cfr.EnsTS().fetch('BC09_NINO34').annualize(months=[12, 1, 2])

We use a chain calling of several methods, including the validation step `.validate()` and the plotting step `.plot_qs()`.

In [ ]:
fig, ax = res.recons['nino3.4'].compare(bc09, timespan=(1874, 1900)).plot(label='recon')
ax.set_xlim(1870, 1900)
ax.set_ylim(-3, 4)
ax.set_ylabel('NINO3.4 [K]')
cfr.showfig(fig)
cfr.savefig(fig, f'./figs/graphem_corr_recon_BC09.pdf')

### Validate the reconstructed fields

In [ ]:
target = cfr.ClimateField().fetch('HadCRUT4.6_GraphEM', vn='tas').get_anom((1951, 1980)).annualize(months=[12, 1, 2])

Calculating and visualizing the correlation coefficient ($r$) between the reconstructed and target fields, we see overall high skills.

In [ ]:
# validate the reconstruction against 20CR
stat = 'corr'

valid_fd = res.recons['tas'].compare(
    target, stat=stat, 
    timespan=(1874, 1900),
)
valid_fd.plot_kwargs.update({'cbar_orientation': 'horizontal', 'cbar_pad': 0.1})

fig, ax = valid_fd.plot(
    title=f'{stat}(recon, obs), mean={valid_fd.geo_mean().value[0,0]:.2f}',
    projection='PlateCarree',
    latlon_range=(-25, 25, 0, 360),
    # latlon_range=(-20, 20, 150, 256),
    plot_cbar=True, plot_proxydb=True, proxydb=job.proxydb,
    plot_proxydb_lgd=True, proxydb_lgd_kws={'loc': 'lower left', 'bbox_to_anchor': (1, 0)},
)

cfr.showfig(fig)
cfr.savefig(fig, f'./figs/graphem_{stat}_recon_obs.pdf')

## Comparison to the LMR/PDA based reconstruction

In [ ]:
res_lmr = cfr.ReconRes('./recons/lmr-real-pages2k', verbose=True)
res_graphem = cfr.ReconRes('./recons/graphem-real-pages2k', verbose=True)

In [ ]:
tas_HadCRUT = cfr.ClimateField().fetch('HadCRUT4.6_GraphEM', vn='tas').get_anom((1951, 1980)).annualize(months=[12, 1, 2])
nino34_bc09 = cfr.EnsTS().fetch('BC09_NINO34').annualize(months=[12, 1, 2])

In [ ]:
res_graphem.valid(
    target_dict={'tas':  tas_HadCRUT, 'nino3.4': nino34_bc09},
    timespan=(1874, 1900), verbose=True,
    stat=['corr', 'CE'],
)

In [ ]:
res_lmr.valid(
    target_dict={'tas':  tas_HadCRUT, 'nino3.4': nino34_bc09},
    timespan=(1874, 1900), verbose=True,
    stat=['corr', 'CE'],
)

In [ ]:
fig, ax = res_graphem.plot_valid(
    target_name_dict={'tas': 'HadCRUT4.6', 'nino3.4': 'BC09'},
    recon_name_dict={'tas': 'GraphEM/tas', 'nino3.4': 'NINO3.4 [K]'},
    valid_fd_kws=dict(
        projection='PlateCarree',
        latlon_range=(-17, 17, 153, 252),
        plot_cbar=True,
    ),
    valid_ts_kws=dict(
        xlim = (1870, 1900),
        ylim = (-3, 4),
    )
)

cfr.visual.add_annotation(ax, fs=[20, 20, 20])

for k, v in fig.items():
    cfr.showfig(v)
    cfr.savefig(v, f'./figs/graphem_{k}.pdf')
    

In [ ]:
fig, ax = res_lmr.plot_valid(
    target_name_dict={'tas': 'HadCRUT4.6', 'nino3.4': 'BC09'},
    recon_name_dict={'tas': 'PDA/tas', 'nino3.4': 'NINO3.4 [K]'},
    valid_fd_kws=dict(
        projection='PlateCarree',
        latlon_range=(-17, 17, 153, 252),
        plot_cbar=True,
    ),
    valid_ts_kws=dict(
        xlim = (1870, 1900),
        ylim = (-3, 4),
    )
)

cfr.visual.add_annotation(ax, fs=[20, 20, 20], start=3)

for k, v in fig.items():
    cfr.showfig(v)
    cfr.savefig(v, f'./figs/lmr_{k}.pdf')